# 🏡 Airbnb Pricing & Occupancy Analytics by Neighbourhood

**Dataset:** [Seattle Airbnb Open Data](https://www.kaggle.com/datasets/airbnb/seattle)  
**Files used:** `listings.csv`, `calendar.csv`  
**Goal:** Benchmark pricing across neighbourhoods, measure seasonal variation, and calculate occupancy rates to identify highest-value areas and listing characteristics.

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Global plot style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'figure.titlesize': 15,
})

ACCENT   = '#FF5A5F'   # Airbnb red
SECONDARY = '#00A699'  # teal
NEUTRAL  = '#767676'

print('Libraries loaded ✅')

---
## 1. Load & Clean `listings.csv`

In [ ]:
# ── Load listings ──────────────────────────────────────────────────────────────
listings = pd.read_csv('listings.csv', low_memory=False)
print(f'listings shape: {listings.shape}')
listings.head(3)

In [ ]:
# ── Clean price column: remove "$" and ",", cast to float ─────────────────────
listings['price'] = (
    listings['price']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .str.strip()
    .replace('', np.nan)
    .astype(float)
)

# Drop listings with missing price or neighbourhood
listings.dropna(subset=['price', 'neighbourhood_cleansed'], inplace=True)

# Remove extreme outliers (price > $1 500 / night — likely data errors)
listings = listings[listings['price'] < 1_500]

print(f'Cleaned listings shape : {listings.shape}')
print(f'Price range            : ${listings["price"].min():.0f} – ${listings["price"].max():.0f}')
print(f'Neighbourhoods         : {listings["neighbourhood_cleansed"].nunique()}')

In [ ]:
# ── Quick data quality snapshot ────────────────────────────────────────────────
key_cols = ['id', 'price', 'neighbourhood_cleansed', 'room_type',
            'instant_bookable', 'number_of_reviews', 'minimum_nights']
listings[key_cols].describe(include='all').T

---
## 2. Load & Clean `calendar.csv`

In [ ]:
# ── Load calendar ──────────────────────────────────────────────────────────────
calendar = pd.read_csv('calendar.csv', low_memory=False)
print(f'calendar shape: {calendar.shape}')
calendar.head(3)

In [ ]:
# ── Convert date & available columns ──────────────────────────────────────────
calendar['date']      = pd.to_datetime(calendar['date'])
calendar['available'] = calendar['available'].map({'t': 1, 'f': 0})  # 1=available, 0=booked

# ── Clean calendar price ───────────────────────────────────────────────────────
calendar['price'] = (
    calendar['price']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)
    .str.strip()
    .replace('', np.nan)
    .astype(float)
)

# Fill NaN calendar prices with per-listing median price
listing_median_price = calendar.groupby('listing_id')['price'].transform('median')
calendar['price']    = calendar['price'].fillna(listing_median_price)

print(f'Date range : {calendar["date"].min().date()} → {calendar["date"].max().date()}')
print(f'Null available : {calendar["available"].isna().sum()}')
print(f'Null price     : {calendar["price"].isna().sum()}')

---
## 3. Occupancy Rate per Listing

In [ ]:
# ── Occupancy = fraction of days where available == 0 (booked) ────────────────
# Days in calendar window per listing (not always exactly 365)
occ = (
    calendar
    .groupby('listing_id')['available']
    .agg(total_days='count', available_days='sum')
    .assign(booked_days=lambda d: d['total_days'] - d['available_days'])
    .assign(occupancy_rate=lambda d: d['booked_days'] / d['total_days'])
    .reset_index()
)

print(f'Listings with occupancy data: {len(occ):,}')
print(f'Avg occupancy rate          : {occ["occupancy_rate"].mean():.1%}')
occ[['occupancy_rate']].describe().T

---
## 4. Merge Occupancy into Listings

In [ ]:
# ── Left-join so every listing is kept even if calendar data is missing ────────
df = listings.merge(
    occ[['listing_id', 'occupancy_rate', 'booked_days', 'total_days']],
    left_on='id', right_on='listing_id',
    how='left'
)

df.drop(columns='listing_id', inplace=True, errors='ignore')

# Encode instant_bookable
df['instant_bookable'] = df['instant_bookable'].map({'t': True, 'f': False, True: True, False: False})

print(f'Final merged shape: {df.shape}')
df[['price', 'occupancy_rate', 'neighbourhood_cleansed', 'room_type']].sample(5)

---
## 5. Neighbourhood Benchmarking

In [ ]:
# ── Aggregate by neighbourhood ─────────────────────────────────────────────────
neigh = (
    df
    .groupby('neighbourhood_cleansed')
    .agg(
        listing_count   = ('id',             'count'),
        median_price    = ('price',          'median'),
        avg_occupancy   = ('occupancy_rate', 'mean'),
    )
    .reset_index()
    .rename(columns={'neighbourhood_cleansed': 'neighbourhood'})
    .sort_values('median_price', ascending=False)
)

# Revenue potential: occupancy_rate × median_price × 365
neigh['revenue_potential'] = neigh['avg_occupancy'] * neigh['median_price'] * 365

print('Top 5 by median price:')
print(neigh.head(5).to_string(index=False))
print('\nBottom 5 by median price:')
print(neigh.tail(5).to_string(index=False))

---
## 6. Visualisation 1 — Top & Bottom 10 Neighbourhoods by Median Price

In [ ]:
top5    = neigh.nlargest(5,  'median_price')
bottom5 = neigh.nsmallest(5, 'median_price')
top10   = pd.concat([top5, bottom5])

fig, ax = plt.subplots(figsize=(12, 6))

colors = [ACCENT if p >= top5['median_price'].min() else SECONDARY
          for p in top10['median_price']]

bars = ax.barh(
    top10['neighbourhood'],
    top10['median_price'],
    color=colors, edgecolor='white', linewidth=0.6
)

# Annotate values
for bar, val in zip(bars, top10['median_price']):
    ax.text(val + 2, bar.get_y() + bar.get_height() / 2,
            f'${val:.0f}', va='center', fontsize=9)

ax.set_xlabel('Median Nightly Price (USD)')
ax.set_title('Top 5 & Bottom 5 Neighbourhoods by Median Nightly Price', fontweight='bold')
ax.axvline(neigh['median_price'].median(), color=NEUTRAL, linestyle='--', linewidth=1,
           label=f'Overall median ${neigh["median_price"].median():.0f}')
ax.legend(fontsize=9)

# Patch labels for legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ACCENT,    label='Top 5  (most expensive)'),
    Patch(facecolor=SECONDARY, label='Bottom 5 (least expensive)'),
]
ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][1:], fontsize=9)

plt.tight_layout()
plt.savefig('viz_neighbourhood_price.png', bbox_inches='tight')
plt.show()
print('Saved → viz_neighbourhood_price.png')

---
## 7. Seasonal Price Analysis — Monthly Median from Calendar

In [ ]:
# ── Monthly median price across the calendar ──────────────────────────────────
calendar['month'] = calendar['date'].dt.to_period('M')

monthly = (
    calendar[calendar['available'] == 0]   # booked nights only
    .groupby('month')['price']
    .median()
    .reset_index()
)
monthly['month_dt'] = monthly['month'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(monthly['month_dt'], monthly['price'],
        color=ACCENT, linewidth=2.5, marker='o', markersize=6, zorder=3)
ax.fill_between(monthly['month_dt'], monthly['price'],
                alpha=0.12, color=ACCENT)

# Shade summer (Jun–Aug)
summer_mask = monthly['month_dt'].dt.month.isin([6, 7, 8])
for _, row in monthly[summer_mask].iterrows():
    ax.axvspan(row['month_dt'] - pd.Timedelta(days=15),
               row['month_dt'] + pd.Timedelta(days=15),
               alpha=0.08, color='gold')

ax.set_xlabel('Month')
ax.set_ylabel('Median Nightly Price (USD)')
ax.set_title('Seasonal Variation: Monthly Median Nightly Price (Booked Nights)', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%d'))
ax.annotate('☀️ Summer peak', xy=(monthly[summer_mask]['month_dt'].iloc[1], 
                                    monthly[summer_mask]['price'].max()),
            xytext=(20, 15), textcoords='offset points',
            fontsize=9, color='darkorange',
            arrowprops=dict(arrowstyle='->', color='darkorange'))

plt.tight_layout()
plt.savefig('viz_seasonal_price.png', bbox_inches='tight')
plt.show()
print('Saved → viz_seasonal_price.png')

---
## 8. Room Type Price Distribution — Boxplot

In [ ]:
# Filter to reasonable price range for a clean plot
df_box = df[df['price'] <= 500].copy()

order = df_box.groupby('room_type')['price'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df_box, x='room_type', y='price', order=order,
    palette=[ACCENT, SECONDARY, '#FFB400', '#6B4FBB'],
    width=0.5, linewidth=1.2, fliersize=2,
    ax=ax
)

# Overlay median labels
for i, rt in enumerate(order):
    med = df_box[df_box['room_type'] == rt]['price'].median()
    ax.text(i, med + 5, f'${med:.0f}', ha='center', fontsize=9, color='black', fontweight='bold')

ax.set_xlabel('Room Type')
ax.set_ylabel('Nightly Price (USD)')
ax.set_title('Price Distribution by Room Type', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%d'))

# Room type share annotation
share = df['room_type'].value_counts(normalize=True) * 100
for i, rt in enumerate(order):
    if rt in share.index:
        ax.text(i, -30, f'{share[rt]:.0f}% of listings',
                ha='center', fontsize=8, color=NEUTRAL)

plt.tight_layout()
plt.savefig('viz_roomtype_price.png', bbox_inches='tight')
plt.show()
print('Saved → viz_roomtype_price.png')

---
## 9. Occupancy vs. Number of Reviews — Scatter

In [ ]:
scatter_df = df.dropna(subset=['occupancy_rate', 'number_of_reviews']).copy()
# Cap reviews for readability
scatter_df = scatter_df[scatter_df['number_of_reviews'] <= 300]

fig, ax = plt.subplots(figsize=(10, 5))

ax.scatter(
    scatter_df['number_of_reviews'],
    scatter_df['occupancy_rate'],
    alpha=0.25, s=18, color=ACCENT, edgecolors='none'
)

# Trend line
z = np.polyfit(scatter_df['number_of_reviews'], scatter_df['occupancy_rate'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 300, 300)
ax.plot(x_line, p(x_line), color=SECONDARY, linewidth=2, label='Trend line')

r = scatter_df['number_of_reviews'].corr(scatter_df['occupancy_rate'])
ax.set_title(f'Occupancy Rate vs. Number of Reviews  (r = {r:.2f})', fontweight='bold')
ax.set_xlabel('Number of Reviews')
ax.set_ylabel('Occupancy Rate')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('viz_occupancy_reviews.png', bbox_inches='tight')
plt.show()
print(f'Pearson r (reviews vs occupancy): {r:.3f}')
print('Saved → viz_occupancy_reviews.png')

---
## 10. Instant Bookable Comparison

In [ ]:
ib = df.dropna(subset=['instant_bookable', 'occupancy_rate']).copy()
ib_summary = (
    ib.groupby('instant_bookable')
    .agg(count=('id', 'count'),
         median_price=('price', 'median'),
         avg_occupancy=('occupancy_rate', 'mean'))
    .rename(index={True: 'Instant Bookable', False: 'Not Instant Bookable'})
)
print(ib_summary.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

bars1 = ax1.bar(ib_summary.index, ib_summary['median_price'],
                color=[ACCENT, SECONDARY], edgecolor='white')
for b in bars1:
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 1,
             f'${b.get_height():.0f}', ha='center', fontsize=10, fontweight='bold')
ax1.set_title('Median Nightly Price', fontweight='bold')
ax1.set_ylabel('USD')
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%d'))

bars2 = ax2.bar(ib_summary.index, ib_summary['avg_occupancy'],
                color=[ACCENT, SECONDARY], edgecolor='white')
for b in bars2:
    ax2.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
             f'{b.get_height():.1%}', ha='center', fontsize=10, fontweight='bold')
ax2.set_title('Average Occupancy Rate', fontweight='bold')
ax2.set_ylabel('Occupancy Rate')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.suptitle('Instant Bookable vs Non-Instant Bookable', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_instant_bookable.png', bbox_inches='tight')
plt.show()
print('Saved → viz_instant_bookable.png')

---
## 11. Bonus: Revenue Potential by Neighbourhood

In [ ]:
# Revenue potential = occupancy_rate × median_price × 365
top10_rev = neigh.nlargest(10, 'revenue_potential')

fig, ax = plt.subplots(figsize=(12, 6))

palette = sns.color_palette('YlOrRd', n_colors=len(top10_rev))
bars = ax.barh(
    top10_rev['neighbourhood'],
    top10_rev['revenue_potential'],
    color=palette[::-1], edgecolor='white'
)

for bar, val in zip(bars, top10_rev['revenue_potential']):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', va='center', fontsize=9)

ax.set_xlabel('Estimated Annual Revenue Potential (USD)')
ax.set_title('Top 10 Neighbourhoods by Revenue Potential\n'
             '(occupancy_rate × median_price × 365)', fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('viz_revenue_potential.png', bbox_inches='tight')
plt.show()

best = neigh.loc[neigh['revenue_potential'].idxmax()]
print(f"\n🏆 Best revenue-potential neighbourhood: {best['neighbourhood']}")
print(f"   Median price: ${best['median_price']:.0f}/night")
print(f"   Avg occupancy: {best['avg_occupancy']:.1%}")
print(f"   Annual revenue potential: ${best['revenue_potential']:,.0f}")
print('Saved → viz_revenue_potential.png')

---
## 12. Minimum Nights vs Price — Correlation

In [ ]:
corr_df = df[df['minimum_nights'] <= 30].dropna(subset=['minimum_nights', 'price'])
r_nights_price = corr_df['minimum_nights'].corr(corr_df['price'])
print(f'Pearson r (minimum_nights vs price): {r_nights_price:.3f}')

# Correlation matrix for key numeric fields
corr_cols = ['price', 'occupancy_rate', 'number_of_reviews',
             'minimum_nights', 'calculated_host_listings_count']
corr_cols = [c for c in corr_cols if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix: Key Listing Metrics', fontweight='bold')
plt.tight_layout()
plt.savefig('viz_correlation_matrix.png', bbox_inches='tight')
plt.show()
print('Saved → viz_correlation_matrix.png')

---
## 13. Key Insights for Hosts & Investors

In [ ]:
# ── Compute insight values dynamically ────────────────────────────────────────
most_expensive = neigh.iloc[0]
least_expensive = neigh.iloc[-1]
best_rev = neigh.loc[neigh['revenue_potential'].idxmax()]

summer_med = monthly[monthly['month_dt'].dt.month.isin([6,7,8])]['price'].median()
winter_med = monthly[monthly['month_dt'].dt.month.isin([12,1,2])]['price'].median()
seasonal_uplift = (summer_med - winter_med) / winter_med * 100

ib_occ   = ib[ib['instant_bookable']==True]['occupancy_rate'].mean()
nib_occ  = ib[ib['instant_bookable']==False]['occupancy_rate'].mean()
ib_price = ib[ib['instant_bookable']==True]['price'].median()
nib_price= ib[ib['instant_bookable']==False]['price'].median()

r_rev_occ = scatter_df['number_of_reviews'].corr(scatter_df['occupancy_rate'])

insights = f"""
╔══════════════════════════════════════════════════════════════════════════════════╗
║       🏠 AIRBNB SEATTLE — 5 KEY INSIGHTS FOR HOSTS & INVESTORS                 ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                  ║
║  1. PRICING GAP IS HUGE: {most_expensive['neighbourhood']} leads at              ║
║     ${most_expensive['median_price']:.0f}/night vs {least_expensive['neighbourhood']}   ║
║     at ${least_expensive['median_price']:.0f}/night — a {(most_expensive['median_price']/least_expensive['median_price']-1)*100:.0f}% spread.            ║
║     Premium locations command 2-3× the rate of entry-level areas.               ║
║                                                                                  ║
║  2. PRICE ≠ REVENUE: {best_rev['neighbourhood']} has the best annual            ║
║     revenue potential (${best_rev['revenue_potential']:,.0f}) thanks to a       ║
║     strong occupancy rate ({best_rev['avg_occupancy']:.1%}), not just high price.║
║     Investors should optimise for revenue potential, not peak price alone.       ║
║                                                                                  ║
║  3. SUMMER PREMIUM ~{seasonal_uplift:.0f}%: Median nightly price in summer is   ║
║     ${summer_med:.0f} vs ${winter_med:.0f} in winter. Hosts should apply dynamic pricing ║
║     (Airbnb Smart Pricing or PriceLabs) to capture peak-season demand.           ║
║                                                                                  ║
║  4. INSTANT BOOK TRADES PRICE FOR VOLUME: Instant Bookable listings are          ║
║     ~{(1-ib_price/nib_price)*100:.0f}% cheaper (${ib_price:.0f} vs ${nib_price:.0f}/night median)  ║
║     but achieve ~{(ib_occ/nib_occ-1)*100:.0f}% higher occupancy ({ib_occ:.1%} vs {nib_occ:.1%}).  ║
║     For new hosts, enabling Instant Book accelerates review accumulation.        ║
║                                                                                  ║
║  5. REVIEWS DRIVE BOOKINGS (r={r_rev_occ:.2f}): Listings with more reviews       ║
║     have meaningfully higher occupancy. Every review matters — prompt guests     ║
║     post-stay and respond to all feedback to build social proof early.           ║
║                                                                                  ║
╚══════════════════════════════════════════════════════════════════════════════════╝
"""
print(insights)

---
## 14. Summary Statistics Table

In [ ]:
summary = pd.DataFrame({
    'Metric': [
        'Total listings analysed',
        'Calendar rows processed',
        'Neighbourhoods covered',
        'Overall median nightly price',
        'Overall avg occupancy rate',
        'Most expensive neighbourhood',
        'Least expensive neighbourhood',
        'Best revenue-potential neighbourhood',
        'Summer vs Winter price premium',
        'Instant Book occupancy advantage',
        'Correlation: reviews → occupancy',
    ],
    'Value': [
        f'{len(df):,}',
        f'{len(calendar):,}',
        f'{neigh.shape[0]}',
        f'${df["price"].median():.0f}',
        f'{df["occupancy_rate"].mean():.1%}',
        f'{most_expensive["neighbourhood"]} (${most_expensive["median_price"]:.0f}/night)',
        f'{least_expensive["neighbourhood"]} (${least_expensive["median_price"]:.0f}/night)',
        f'{best_rev["neighbourhood"]} (${best_rev["revenue_potential"]:,.0f}/yr)',
        f'+{seasonal_uplift:.0f}%',
        f'+{(ib_occ/nib_occ-1)*100:.0f}%',
        f'r = {r_rev_occ:.2f}',
    ]
})
summary.set_index('Metric', inplace=True)
summary

---
*Project complete. All visualisations saved as `.png` files in the working directory.*